# nano Beatrix — the local two-position pointer screen: the supervised head told where the record is at both address bytes

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_pointer2.ipynb)

**The question**: Alhena (Castor's body turned by the anchored multi-position rotary) answers a question exactly when it is handed the address it asks for, and dozens of independently trained arms then merge under the gate with nothing lost. The address has to come from the model itself, through the index it writes. The last screen (Alhena-P, `colab/nano_pointer.ipynb`) told one attention head (block 4, head 0) where the record is at each index entry: the head looked at the right record at the end of every run, and the model's own index locked on only 1 of 6 seeds. Two readings of those runs say why. First, the term's gradient reached the whole trunk, and on five of six seeds it grew an early look in block 1 that matches the key's last two bytes (which the separator's own input already carries); the index formed on that look and stopped where a match on two bytes stops. Second, an index entry writes its address in two bytes at two positions, and the term supervised only the first: on the seed where the supervised head was the lookup, the first digit was exact and the second was copied from a neighbouring record.

**Alhena-P2** makes two changes to Alhena-P. The term asks the head's share of attention on the record at both of the entry's address positions, and its gradient is kept in that head's own query and key (its query and key are formed from block 4's input with the input detached, so nothing below the head is paid to build a shortcut):

    L = L_text + L_address_head + .1 * mean over entries and both positions of ( - log [ attention mass of block 4, head 0, on the record ] )

with no new parameter and the forward pass unchanged. Everything else is Alhena-P's recipe and read (the value-path tables, the address head, 24,000 steps); every run also reads its index on 64 key-hard rows, a read that leaves the training untouched.

| part | what | why |
|---|---|---|
| P2 s1-s6 | Alhena-P with the term at both address positions and its gradient kept in the head | does the model's own lookup (its written index) lock on at least 5 of 6 seeds |
| P s7, s8 | Alhena-P again on two new seeds (series `pointerrep`) | the replication arm: Alhena-P's lock rate over runs |
| members | the scale test's 48 real arms + 48 toy members (after the last block) and 16 real arms (after block 3), each trained alone, on the first two P2 cores (in seed order) that lock | the request = the model's own written index (free-running); merged under w_i = sinh(u_i) / (1 + sum_j (cosh(u_j) - 1)) |

**The cells**: C1 pins the trainer and the arm library and holds the decision block; C2 builds the data, runs 200 steps of Alhena-P2 on this card for the price line (a projection past 1.5x the plan stops for a re-plan) and probes the push path; C3 trains P2 s1; C4 trains the members on P2 s1 if its written index is right on at least .98 of its entries (free-running) and its answers with the true address reach .95; C5 trains P2 s2, P2 s3, P s7, P2 s4, P2 s5, P s8 and P2 s6; C6 trains the members on the next P2 cores in seed order while fewer than two cores carry them; C7 renders the tables. Every finished run ships to [AbstractPhil/alephllm-nano-beatrix-training](https://huggingface.co/AbstractPhil/alephllm-nano-beatrix-training) (`bed/reports/`, `bed/weights/<run>/` with the address tables in `value_addr.safetensors`, `bed/scale_members/`).

**Before C1**: the notebook's `HF_TOKEN` secret (write access to that repo) with notebook access on. Run C1, then Runtime > Restart session and run all. A file named `STOP` in `/content/out` ends the session between runs; a restarted session resumes where it stopped. The code and its tests: [AbstractEyes/geolip-nano-beatrix-trainer](https://github.com/AbstractEyes/geolip-nano-beatrix-trainer) (`geolip/nano_beatrix/pointer.py`, `bed_train.py`).

In [ ]:
# C1 — pinned installs + THE DECISION BLOCK. RESTART the session after the first run of this cell (Runtime > Restart session and run all).
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.79" "amoe-lora @ git+https://github.com/AbstractEyes/amoe-lora@v0.2.12"
import torch, amoe
import geolip.nano_beatrix as N
print('geolip.nano_beatrix', N.__version__, '| amoe', amoe.__version__, '| torch', torch.__version__, '| card',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 79), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart session, then rerun from C1')
assert amoe.__version__ == '0.2.12', 'the arm module of record is amoe-lora 0.2.12'
assert torch.cuda.is_available(), 'no card: Runtime > Change runtime type > GPU'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN            # the runner and the shipper read it from the environment; the value is never printed

# ============ THE DECISION BLOCK — every setting set from the plan, with its source beside it ============
OUT = '/content/out'                                   # the session folder: the corpus, the records, the weights, the members, runner.log
REPO = 'AbstractPhil/alephllm-nano-beatrix-training'  # every finished run ships here; tokenless = nothing ships
CORPUS_BYTES = 160e6                                   # fineweb-edu text for the bed (the scale test's corpus)
BED = dict(                                            # the scale test's Alhena recipe at 24,000 steps, verbatim
    ctx=4096, mb=8, steps=24000,                       # 24,000 steps x 8 rows x 4,096 bytes; pure Adam, lr 1e-3, 100 warm-up steps, the cosine to a tenth
    frames='dyn', plant=1.0,
    sizes='32,128,256,512', frame_records=4, nest=0.3, overlap=0.0, ask='any', register='caption', tags='mode',
    log_every=100,
)
GRID_ROWS = dict(grid='ag', grid_p=0.5,                # half of every batch's rows are addressed grid rows
                 head='digit', head_lam=1.0,           # the per-digit address head at the questions' SEPs (the scale test's term)
                 trace_every=2000,                     # the formation trace every 2,000 steps (the pointer's masses read at each point)
                 hard_read=1)                          # the index on 64 key-hard rows at the end (a read: the training unchanged)
WIDE = dict(rotary_content=4)                          # ALHENA-W: 4 position / 4 content / 8 address pairs
VALUE = dict(value_addr=1)                             # ALHENA-V: the value-path tables, born zero (the name carries _va1)
POINTER = dict(ptr_lam=0.1, ptr_block=4, ptr_head=0)  # ALHENA-P: block 4 of 6, head 0, lambda .1 (the pointer screen's rule)
LOCAL2 = dict(ptr_pos='both', ptr_stop=1)              # ALHENA-P2: both address positions; the gradient kept in the head (name: p2sg)
Q = lambda s: (f'P2 s{s}', dict(BED, **GRID_ROWS, **WIDE, **VALUE, **POINTER, **LOCAL2, arms='alhena-both', seeds=str(s), series='pointer2'))
R = {s: (f'P s{s}', dict(BED, **GRID_ROWS, **WIDE, **VALUE, **POINTER, arms='alhena-both', seeds=str(s), series='pointerrep')) for s in (7, 8)}
TAIL = '_L24000_ctx4096_frdyn_cap_mode_sz32-128-256-512_r4_n30_o0_askany_gag50'
TAG_Q = lambda s: f'alhena-both_s{s}{TAIL}_cp4_va1_pt10b4h0p2sg_ahdigit1_pointer2'
MIN_INDEX, MIN_ANSWERS = 0.98, 0.95                    # the plan's member rule: the written index (free-running) >= .98, answers >= .95
N_MEMBER_CORES = 2                                     # the first two P2 cores, in seed order, that pass the rule
LAST_STEPS = 16000                                     # the scale test's member budget, verbatim
MID_STEPS = 32000                                      # batch steps at the mid site (16,000 own steps a member)
READ_BATCHES, CALIB_BATCHES = 64, 32                   # the scale test's read and calibration rows (8 rows a batch)
SMOKE_STEPS = 200                                      # C2's short run of Alhena-P2 on this card for the price line; its record stays local
PRICE_STEPS = 200                                      # the member price lines (each trainer on throwaway stacks)
PLAN_MIN = 17.4                                        # the planned minutes of one 24,000-step core on this card (Alhena-P measured 17.2; the local term est. < 1%)
PLAN_MIN_MEM = 18.8                                    # the planned training minutes of one core's members (4.9 last + 13.8 mid measured)
STOP_FACTOR = 1.5                                      # a projection past 1.5x the plan stops the session for a re-plan
PARALLEL = 1                                           # ONE run at a time: one 9M craft at 8 x 4,096 saturates the card
print('decision block: P2 s1-s6 + P s7, s8 | the member rule', MIN_INDEX, MIN_ANSWERS, '| out', OUT,
      '| ships to', REPO if HF_TOKEN else 'nothing (tokenless)')

In [ ]:
# C2 (P) — PREFLIGHT: the data, the plan, 200 steps of Alhena-P2 on this card (the price line), the push probe. No part trains.
import json, os, time, torch
from geolip.nano_beatrix import bed_corpus, bed_runner
if not os.path.exists(f'{OUT}/bed/bed_corpus.bin'):
    bed_corpus.build(f'{OUT}/bed', CORPUS_BYTES)
HELD = f'{OUT}/bed/heldout_fineweb_edu_head2048.jsonl'
print('corpus', os.path.getsize(f'{OUT}/bed/bed_corpus.bin') / 1e6, 'MB; held-out head', os.path.exists(HELD))
todo = 0
for name, spec in [Q(s) for s in (1, 2, 3, 4, 5, 6)] + [R[7], R[8]]:
    p = bed_runner.run(spec, out=OUT, dry=True)
    todo += p['runs'] - p['done_before']
smoke = dict(Q(1)[1], steps=SMOKE_STEPS, log_every=20, trace_every=100, read_batches=4, calib_batches=2, hard_batches=1)
res = bed_runner.run(smoke, out=OUT)
assert res['failed'] is None, 'the short run failed: read the lines above before any part'
last = [s for s in (bed_runner.parse_step(l) for l in open(f'{OUT}/runner.log', encoding='utf-8')) if s][-1]
per_run = BED['steps'] * BED['ctx'] * BED['mb'] / (last['tok_s'] * 1e3) / 60
PRICE_OK = per_run <= STOP_FACTOR * PLAN_MIN
print(f"this card: {last['tok_s']}k tok/s at {BED['ctx']} x {BED['mb']} -> one {BED['steps']:,}-step run about {per_run:.1f} min (planned "
      f"{PLAN_MIN} min); the {todo} runs left about {per_run * todo / 60:.1f} h (+ a few minutes of end reads a run, + the members)"
      + ('' if PRICE_OK else f' -> PAST {STOP_FACTOR}x THE PLAN: C3 will not start; re-plan first'))
PUSH_OK = False
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    path = 'bed/preflight/push_probe_pointer2.json'
    body = json.dumps({'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'card': torch.cuda.get_device_name(0),
                       'tok_s': last['tok_s'], 'parts': ['P2 s1-s6', 'P s7', 'P s8']}).encode()
    api.upload_file(path_or_fileobj=body, path_in_repo=path, repo_id=REPO, commit_message='nano pointer2: push probe')
    PUSH_OK = path in set(api.list_repo_files(REPO))
    print('push probe', 'READ BACK OK' if PUSH_OK else 'NOT VISIBLE after upload', path)
else:
    print('TOKENLESS: no probe written; the cells below run and ship nothing')

In [ ]:
# C3 (T) — P2 s1, resume-first (runner_done.txt in OUT); a STOP file in OUT starts no new run; every finished run ships to REPO as it
# lands (one background shipper: the card never waits on the Hub).
PARTS_NOW = [Q(1)]
from geolip.nano_beatrix import bed_runner
assert PRICE_OK, 'the price line projected past the plan: re-plan before any part trains'
push = REPO if (HF_TOKEN and PUSH_OK) else ''
shipper = globals().get('shipper') or (bed_runner._Shipper(push, OUT, print) if push else None)
if shipper:
    for rec in bed_runner.pending_records(OUT):
        shipper.submit(rec)
done = []
for name, spec in PARTS_NOW:
    if os.path.exists(f'{OUT}/STOP'):
        print('STOP present: no new part starts')
        break
    print(f'=================== {name} ===================')
    res = bed_runner.run(spec, out=OUT, push=push, parallel=PARALLEL, shipper=shipper)
    done.append((name, res))
    if res['failed'] is not None or res['stopped']:
        print('the part did not finish:', name, {k: v for k, v in res.items() if k != 'ran'})
        break
if shipper:
    shipper.drain(print)
    left = [os.path.basename(p)[4:-5] for p in bed_runner.pending_records(OUT)]
    print('UNSHIPPED:', left if left else 'none')
print('session:', [(n, r['failed'], r['stopped'], len(r['ran'])) for n, r in done])

In [ ]:
# C4 (T) — THE MEMBERS on P2 s1: if its own end read writes its index right on >= .98 of the entries (free-running) and answers >= .95
# with the true address, it carries the scale test's members verbatim; the request is the core's own written index (the two-pass read, the
# question-head set: LAYOUT, INDEX, INDEXC, INDEXF, HEADPF, HEADPFQ999, HEADI, HEADIF, G2, G0). Fewer than two member cores -> C6 takes
# the next P2 cores in seed order.
import json, time
from geolip.nano_beatrix import rotary_scale as rs
BED_DIR = f'{OUT}/bed'

def _record(tag):
    p = f'{BED_DIR}/bed_{tag}.json'
    return json.load(open(p, encoding='utf-8')) if os.path.exists(p) else None

def _qualifies(tag):
    r = _record(tag)
    if r is None:
        return None, 'no record in this session'
    g = r['grid']['read']
    fr = g.get('free_running') or {}
    ans = g['true']['whole']
    look = fr.get('indexf_right', 0.0)
    ok = look >= MIN_INDEX and ans >= MIN_ANSWERS
    return ok, f"its written index (free-running) {look:.4f}, answers with the true address {ans:.4f}"

def run_members(cores):
    owned, never = rs.owned_sets()
    packed = rs.load_packed(BED_DIR)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    core = rs.Core(f'{BED_DIR}/{cores[0][0]}', 'cuda')
    print('the core\'s address form:', 'one byte' if core.abytes == 1 else 'two digits')
    cc = rs.compile_check(core, packed, owned, never)
    print('compiled = eager at step 1 (largest difference over the largest value):', cc)
    assert all(v['write'] < 1e-4 and v['grad'] < 1e-3 for v in cc.values()), 'the compiled writes are not the eager writes: no member trains'
    pr = rs.price(core, packed, owned, never, steps=PRICE_STEPS)
    per_core = (pr['last_ms_per_step'] * LAST_STEPS + pr['mid_ms_per_step'] * MID_STEPS) / 60000
    print(f"this card: last site {pr['last_ms_per_step']:.1f} ms a step, mid site {pr['mid_ms_per_step']:.1f} ms a step -> a core's members "
          f"about {per_core:.0f} min of training (planned {PLAN_MIN_MEM}) + the reads")
    assert per_core <= STOP_FACTOR * PLAN_MIN_MEM, 'the members priced past the plan: re-plan before any member trains'
    del core
    torch.cuda.empty_cache()
    ship = rs.Shipper(REPO) if (HF_TOKEN and PUSH_OK) else None
    t_all = time.time()
    out = []
    for n, (tag, seed) in enumerate(cores, 1):
        if os.path.exists(f'{OUT}/STOP'):
            print('STOP present: no new core starts')
            break
        print(f'=================== members {n}/{len(cores)} {tag} ===================')
        t0 = time.time()
        res = rs.run_core(f'{BED_DIR}/{tag}', f'{OUT}/members', seed, BED_DIR, HELD, last_steps=LAST_STEPS, mid_steps=MID_STEPS,
                          read_batches=READ_BATCHES, calib_batches=CALIB_BATCHES, shipper=ship, forms=('find',))
        out.append(res)
        torch.cuda.empty_cache()
        print(f'{tag}: {(time.time() - t0) / 60:.1f} min; {(time.time() - t_all) / 60:.1f} min spent; {len(cores) - n} cores left')
    if ship:
        print('ships drained' if ship.drain() else 'ships still retrying in the background')
    return out
MEMBER_CORES = []
ok, why = _qualifies(TAG_Q(1))
print(f'P2 s1: {why} -> ' + ('its members train' if ok else 'NOT HERE (C6 takes the next P2 core that passes)'))
if ok:
    MEMBER_CORES.append((TAG_Q(1), 1))
done_m = run_members(MEMBER_CORES) if MEMBER_CORES else []
print('members:', [r['name'] for r in done_m], '| member cores still wanted:', N_MEMBER_CORES - len(MEMBER_CORES))

In [ ]:
# C5 (T) — THE OTHER CORES: P2 s2, P2 s3, then the replication arm between the P2 seeds (P s7, P2 s4, P2 s5, P s8, P2 s6); one run at a
# time, resume-first, shipped as each lands.
PARTS_NOW = [Q(2), Q(3), R[7], Q(4), Q(5), R[8], Q(6)]
from geolip.nano_beatrix import bed_runner
assert PRICE_OK, 'the price line projected past the plan: re-plan before any part trains'
push = REPO if (HF_TOKEN and PUSH_OK) else ''
shipper = globals().get('shipper') or (bed_runner._Shipper(push, OUT, print) if push else None)
if shipper:
    for rec in bed_runner.pending_records(OUT):
        shipper.submit(rec)
done = []
for name, spec in PARTS_NOW:
    if os.path.exists(f'{OUT}/STOP'):
        print('STOP present: no new part starts')
        break
    print(f'=================== {name} ===================')
    res = bed_runner.run(spec, out=OUT, push=push, parallel=PARALLEL, shipper=shipper)
    done.append((name, res))
    if res['failed'] is not None or res['stopped']:
        print('the part did not finish:', name, {k: v for k, v in res.items() if k != 'ran'})
        break
if shipper:
    shipper.drain(print)
    left = [os.path.basename(p)[4:-5] for p in bed_runner.pending_records(OUT)]
    print('UNSHIPPED:', left if left else 'none')
print('session:', [(n, r['failed'], r['stopped'], len(r['ran'])) for n, r in done])

In [ ]:
# C6 (T) — THE SUBSTITUTION RULE (the plan's, written before any run): while fewer than two P2 cores carry members, the next P2 core in
# seed order (P2 s2, then P2 s3 ... P2 s6) that passes the same rule carries them.
import json, time
from geolip.nano_beatrix import rotary_scale as rs
BED_DIR = f'{OUT}/bed'

def _record(tag):
    p = f'{BED_DIR}/bed_{tag}.json'
    return json.load(open(p, encoding='utf-8')) if os.path.exists(p) else None

def _qualifies(tag):
    r = _record(tag)
    if r is None:
        return None, 'no record in this session'
    g = r['grid']['read']
    fr = g.get('free_running') or {}
    ans = g['true']['whole']
    look = fr.get('indexf_right', 0.0)
    ok = look >= MIN_INDEX and ans >= MIN_ANSWERS
    return ok, f"its written index (free-running) {look:.4f}, answers with the true address {ans:.4f}"

def run_members(cores):
    owned, never = rs.owned_sets()
    packed = rs.load_packed(BED_DIR)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    core = rs.Core(f'{BED_DIR}/{cores[0][0]}', 'cuda')
    print('the core\'s address form:', 'one byte' if core.abytes == 1 else 'two digits')
    cc = rs.compile_check(core, packed, owned, never)
    print('compiled = eager at step 1 (largest difference over the largest value):', cc)
    assert all(v['write'] < 1e-4 and v['grad'] < 1e-3 for v in cc.values()), 'the compiled writes are not the eager writes: no member trains'
    pr = rs.price(core, packed, owned, never, steps=PRICE_STEPS)
    per_core = (pr['last_ms_per_step'] * LAST_STEPS + pr['mid_ms_per_step'] * MID_STEPS) / 60000
    print(f"this card: last site {pr['last_ms_per_step']:.1f} ms a step, mid site {pr['mid_ms_per_step']:.1f} ms a step -> a core's members "
          f"about {per_core:.0f} min of training (planned {PLAN_MIN_MEM}) + the reads")
    assert per_core <= STOP_FACTOR * PLAN_MIN_MEM, 'the members priced past the plan: re-plan before any member trains'
    del core
    torch.cuda.empty_cache()
    ship = rs.Shipper(REPO) if (HF_TOKEN and PUSH_OK) else None
    t_all = time.time()
    out = []
    for n, (tag, seed) in enumerate(cores, 1):
        if os.path.exists(f'{OUT}/STOP'):
            print('STOP present: no new core starts')
            break
        print(f'=================== members {n}/{len(cores)} {tag} ===================')
        t0 = time.time()
        res = rs.run_core(f'{BED_DIR}/{tag}', f'{OUT}/members', seed, BED_DIR, HELD, last_steps=LAST_STEPS, mid_steps=MID_STEPS,
                          read_batches=READ_BATCHES, calib_batches=CALIB_BATCHES, shipper=ship, forms=('find',))
        out.append(res)
        torch.cuda.empty_cache()
        print(f'{tag}: {(time.time() - t0) / 60:.1f} min; {(time.time() - t_all) / 60:.1f} min spent; {len(cores) - n} cores left')
    if ship:
        print('ships drained' if ship.drain() else 'ships still retrying in the background')
    return out
want = N_MEMBER_CORES - len(globals().get('MEMBER_CORES', []))
SUB = []
for s in (2, 3, 4, 5, 6):
    if len(SUB) >= want:
        break
    ok, why = _qualifies(TAG_Q(s))
    print(f'P2 s{s}: {why} -> ' + ('its members train' if ok else 'not a member core'))
    if ok:
        SUB.append((TAG_Q(s), s))
done_s = run_members(SUB) if SUB else []
print('substitute members:', [r['name'] for r in done_s] or ('none needed' if want <= 0 else 'none (no other P2 core passed)'))

In [ ]:
# C7 — THE TABLES: the cores (bits, the frame copy, the index both ways, the key-hard rows' index both ways, the answers, the supervised
# head's mass at steps 2,000 / 4,000 / the end read at the SEP and, on P2, at the first address byte, minutes) and the members (per core
# and stack: solo against bare, the nine rest cells keeping all three marks per source, the controls, the largest leak of the written
# index, the mid site, the text bits). Run any time.
import glob, json
from IPython.display import Markdown, display
rows = ['| run | bits | copy 32 / 128 / 256 / 512 | index TF / FR | key-hard rows TF / FR | answers true / via index | pointer mass 2k / 4k / read (SEP; first address byte) | min |',
        '|---|---|---|---|---|---|---|---|']
for p in sorted(glob.glob(f'{OUT}/bed/bed_alhena-both_*_L24000_*.json')):
    r = json.load(open(p, encoding='utf-8'))
    if not r.get('grid'):
        continue
    ff = r.get('fetch_by_frame') or {}
    copy = ' / '.join(f"{ff[f'data/{s}/depth0']['full_value']:.3f}" if f'data/{s}/depth0' in ff else '-' for s in (32, 128, 256, 512))
    g = r['grid']['read']
    fr = g.get('free_running') or {}
    hd = g.get('hard')
    hard = f"{hd['index_right']:.4f} / {hd['indexf_right']:.4f}" if hd else '-'
    via = f"{g['true']['whole']:.4f} / {fr.get('indexf_whole', float('nan')):.4f}"
    pt_ = r['grid'].get('ptr')
    if pt_:
        tm = {t['step']: (t.get('ptr_mass'), t.get('ptr_mass2')) for t in r['grid']['trace']}
        f_ = lambda v: '-' if v is None else f'{v:.3f}'
        ptr = (f"{f_(tm.get(2000, (None,))[0])} / {f_(tm.get(4000, (None,))[0])} / {f_(pt_['mass_read'])}"
               + (f"; {f_(tm.get(2000, (None, None))[1])} / {f_(tm.get(4000, (None, None))[1])} / {f_(pt_.get('mass_read2'))}" if 'mass_read2' in pt_ else ''))
    else:
        ptr = '-'
    _n = os.path.basename(p)[4:-5]
    rows.append(f"| {_n.split('_ctx')[0]} {_n.split('gag50_')[-1]} | {r['heldout_bpb']:.4f} | {copy} | {g['true']['index_right']:.4f} / "
                f"{fr.get('indexf_right', float('nan')):.4f} | {hard} | {via} | {ptr} | {r['train_minutes']:.1f} |")
display(Markdown('\n'.join(rows)))
rows = ['| core | stack | solo / bare owned | INDEXF | HEADPF | LAYOUT | G2 m48 | INDEXC m48 Q | G0 m48 P/Q | largest leak / own (INDEXF) | mid INDEXF U5/8/10 | G0 text bits |',
        '|---|---|---|---|---|---|---|---|---|---|---|---|']
for d in sorted(glob.glob(f'{OUT}/members/*_members_*')):
    if not os.path.exists(f'{d}/last_find.json'):
        continue
    L = json.load(open(f'{d}/last_find.json', encoding='utf-8'))
    M = json.load(open(f'{d}/mid_find.json', encoding='utf-8')) if os.path.exists(f'{d}/mid_find.json') else None
    rd = L['read']
    for st, r in rd['stacks'].items():
        c = r['cells']
        allm = lambda x: all(x[k] for k in ('PROTECTED', 'QUIET', 'EXACT'))
        nine = lambda s: f"{sum(allm(c[f'{s}_U{u}_REST_m{m}']) for u in (5, 8, 10) for m in (16, 32, 48))}/9" if f'{s}_U5_REST_m16' in c else '-'
        leak = max(c[f'INDEXF_U{u}_REST_m{m}']['leak_over_own_write'] for u in (5, 8, 10) for m in (16, 32, 48))
        mid = '-'
        if M and st == 'arms':
            mc = M['read']['cells']
            mid = ' '.join(str(allm(mc[f'INDEXF_U{u}_REST_m16'])) if f'INDEXF_U{u}_REST_m16' in mc else '-' for u in (5, 8, 10))
        g0 = L['text_bits'].get(f'G0_{st}_minus_bare')
        rows.append(f"| {L['core'][:30]} | {st} | {r['solo']['ONE']['48']:.3f} / {rd['bare']['owned']:.3f} | {nine('INDEXF')} | {nine('HEADPF')} | "
                    f"{nine('LAYOUT')} | {allm(c['G2_m48'])} | {c['INDEXC_U5_REST_m48']['QUIET']} | "
                    f"{c['G0_m48']['PROTECTED']}/{c['G0_m48']['QUIET']} | {leak:.1e} | {mid} | {g0:+.3f} |")
display(Markdown('\n'.join(rows)))